# 10 — System Profiles: Per-System Deep Dives

Detailed quality profile analysis for each of the four case systems.
Covers daily metric trajectories, dimension strengths/weaknesses,
and the paper's key per-system findings.

| System | Domain | LLM | Composite |
|--------|--------|-----|-----------|
| S1 | Customer Support | GPT-4o | 7.23 |
| S2 | AI Code Assistant | Claude 3.5 Sonnet FT | 7.68 |
| S3 | Document Summarisation & QA | Gemini 1.5 Pro | 8.02 |
| S4 | Medical Triage | Llama 3.1 70B (self-hosted) | 8.15 |


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

SYSTEM_META = {
    'S1': {'dir':'S1_Customer_Support_Chatbot',     'llm':'GPT-4o',               'composite':7.23},
    'S2': {'dir':'S2_AI_Code_Assistant_IDE_Plugin',  'llm':'Claude 3.5 Sonnet FT', 'composite':7.68},
    'S3': {'dir':'S3_Document_Summarization_and_QA', 'llm':'Gemini 1.5 Pro',       'composite':8.02},
    'S4': {'dir':'S4_Medical_Triage_Assistant',       'llm':'Llama 3.1 70B',        'composite':8.15},
}
DIMS = ['FC','RO','SF','SS','TI','IQ']
DIM_SCORES = {
    'S1':{'FC':7.8,'RO':6.2,'SF':8.1,'SS':7.4,'TI':5.6,'IQ':8.3},
    'S2':{'FC':8.9,'RO':7.8,'SF':7.3,'SS':8.8,'TI':6.2,'IQ':7.1},
    'S3':{'FC':8.2,'RO':6.8,'SF':9.1,'SS':7.9,'TI':7.5,'IQ':8.6},
    'S4':{'FC':7.1,'RO':8.3,'SF':8.6,'SS':9.2,'TI':8.9,'IQ':6.8},
}

daily = {}
for sid, meta in SYSTEM_META.items():
    df = pd.read_csv(f'../data/raw/{meta["dir"]}/metric_snapshots/daily_metrics.csv')
    df['date'] = pd.to_datetime(df['date'])
    daily[sid] = df

print("Daily metric snapshots loaded:")
for sid, df in daily.items():
    print(f"  {sid}: {len(df)} rows, columns: {list(df.columns)}")

Daily metric snapshots loaded:
  S1: 92 rows, columns: ['date', 'FC-1', 'SF-1', 'SF-3', 'SS-4', 'TI-1', 'TI-4', 'IQ-4', 'week']
  S2: 92 rows, columns: ['date', 'FC-1', 'SF-1', 'SF-3', 'SS-4', 'TI-1', 'TI-4', 'IQ-4', 'week']
  S3: 92 rows, columns: ['date', 'FC-1', 'SF-1', 'SF-3', 'SS-4', 'TI-1', 'TI-4', 'IQ-4', 'week']
  S4: 92 rows, columns: ['date', 'FC-1', 'SF-1', 'SF-3', 'SS-4', 'TI-1', 'TI-4', 'IQ-4', 'week']


## 1. Dimension scores — all four systems side-by-side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
N = len(DIMS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

for i, (sid, meta) in enumerate(SYSTEM_META.items()):
    ax = axes[i]
    scores = [DIM_SCORES[sid][d] for d in DIMS] + [DIM_SCORES[sid][DIMS[0]]]
    ax.plot(angles, scores, 'o-', linewidth=2, color=colors[i])
    ax.fill(angles, scores, alpha=0.15, color=colors[i])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(DIMS, size=11)
    ax.set_ylim(0, 10)
    ax.set_yticks([5,7,9])
    ax.axhline(y=7.77, color='grey', linestyle='--', linewidth=0.7)
    title = f"{sid} — {meta['llm']}\nComposite: {meta['composite']}"
    ax.set_title(title, size=11, pad=15)

plt.suptitle('System Quality Profiles — S1 to S4 (dashed = study mean 7.77)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Daily FC-1 and SF-3 trajectories

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
colors = {'S1':'#4C72B0','S2':'#DD8452','S3':'#55A868','S4':'#C44E52'}

for sid, df in daily.items():
    if 'FC-1' in df.columns:
        axes[0].plot(df['date'], df['FC-1'], label=sid, color=colors[sid], linewidth=1.5, alpha=0.85)
axes[0].axhline(0.85, color='red', linestyle='--', linewidth=1, label='Threshold (0.85)')
axes[0].set_ylabel('FC-1 Task Accuracy'); axes[0].set_title('FC-1 Daily Trajectory (Oct–Dec 2024)')
axes[0].legend(); axes[0].grid(alpha=0.2)

for sid, df in daily.items():
    if 'SF-3' in df.columns:
        axes[1].plot(df['date'], df['SF-3'], label=sid, color=colors[sid], linewidth=1.5, alpha=0.85)
axes[1].axhline(2.0, color='red', linestyle='--', linewidth=1, label='Threshold (2.0)')
axes[1].set_ylabel('SF-3 Hallucination Rate /1K'); axes[1].set_title('SF-3 Daily Trajectory')
axes[1].legend(); axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Per-system key findings (paper Section 6.1)

In [ ]:
findings = {
    'S1': [
        "Lowest TI score (5.6) across all systems — transparency gap confirmed by interviews",
        "SF-2 attribution coverage gap: 22% of RAG responses lack source citations",
        "IQ-2/IQ-1 r=0.74 most pronounced in S1 — latency spikes predict availability incidents",
        "11 incidents; mean QALIS lag 1.4h vs baseline 5.2h (73% improvement)",
    ],
    'S2': [
        "Highest FC score (8.9) — FC-3 Pass@k=0.91 leads all systems on code correctness",
        "TI-3 not applicable (IDE plugin has no explanation UI) — excluded from TI composite",
        "IQ-3 excluded (streaming FIM cost tracking not aligned with QALIS formula)",
        "10 incidents; mean QALIS lag 1.1h (shortest across all systems)",
    ],
    'S3': [
        "Highest SF score (9.1) — long-context grounding (1M ctx) reduces hallucination rate",
        "SF-3 = 0.8/1K tokens — lowest hallucination rate across all four systems",
        "GDPR override: SS-2 ≤ 0.0005 (vs default 0.001); extra EU PII entity categories",
        "10 incidents; IQ-2 threshold relaxed to 4000ms (justified for long-doc processing)",
    ],
    'S4': [
        "Highest composite (8.15) despite lowest IQ (6.8) — regulatory pressure drives quality",
        "Highest TI (8.9) and SS (9.2) — HIPAA/CQC/EU AI Act as forcing function",
        "Human-in-loop: ~18% of queries routed to clinical staff (confidence < 75%)",
        "11 incidents; strictest thresholds (SF-3≤1.0, RO-2≥0.99, SS-2≤0.0001)",
    ],
}
for sid, items in findings.items():
    meta = SYSTEM_META[sid]
    print(f"\n{'='*55}")
    print(f"  {sid} — {meta['llm']}  (composite: {meta['composite']})")
    print(f"{'='*55}")
    for item in items:
        print(f"  • {item}")


  S1 — GPT-4o  (composite: 7.23)
  • Lowest TI score (5.6) across all systems — transparency gap confirmed by interviews
  • SF-2 attribution coverage gap: 22% of RAG responses lack source citations
  • IQ-2/IQ-1 r=0.74 most pronounced in S1 — latency spikes predict availability incidents
  • 11 incidents; mean QALIS lag 1.4h vs baseline 5.2h (73% improvement)

  S2 — Claude 3.5 Sonnet FT  (composite: 7.68)
  • Highest FC score (8.9) — FC-3 Pass@k=0.91 leads all systems on code correctness
  • TI-3 not applicable (IDE plugin has no explanation UI) — excluded from TI composite
  • IQ-3 excluded (streaming FIM cost tracking not aligned with QALIS formula)
  • 10 incidents; mean QALIS lag 1.1h (shortest across all systems)

  S3 — Gemini 1.5 Pro  (composite: 8.02)
  • Highest SF score (9.1) — long-context grounding (1M ctx) reduces hallucination rate
  • SF-3 = 0.8/1K tokens — lowest hallucination rate across all four systems
  • GDPR override: SS-2 ≤ 0.0005 (vs default 0.001); extra EU 